# Split test: full K-fold members, or one base model fine-tuned on more folds?

Two ways to train the ensemble with `znn.train_from_catalog`, both with K-fold data cuts (member k validates on fold k and trains on the others), every member on its own noise realization of the training photometry, and the pop-cosmos start:

| ensemble | members | how each member is trained |
|---|---|---|
| `kfold5` | 5 | in full from the pop-cosmos start, up to `EPOCHS` at the Adam default learning rate |
| `frombase_kfold10` | 10 | a copy of one base model (trained first on all the objects with the photometry as given) fine-tuned on its fold at `FINETUNE_LR` for at most `FINETUNE_EPOCHS` |

`from_base` is meant to be cheaper per member, so it can afford more of them; the question is whether 10 fine-tuned members score as well as 5 fully trained ones, and at what training cost. Note that the base model has seen every member's validation fold, so the fine-tuned members' early stopping is not on unseen data.

The saved `baseline` ensemble (5 K-fold members from the 5 pop-cosmos starts, catalogue photometry, no noise realizations) is re-evaluated as a reference. The previous split test (commit f881508) found that retraining with identical settings moves the scatter by about 5% and KS by about 15%: treat differences smaller than that as noise. That test also found that K-fold cuts beat one shared cut, and that one pop-cosmos start is as good as five.

Everything is on the configuration that degraded most (task set 3, cardinal, 10yr), scored on the same held-out objects (the `VAL_FRAC`, `SEED` split of the task-set notebooks, never used in training) and predicted the same way: `znn.predict_from_catalog` with `N_SAMPLES` noise draws from the same seed, giving a Gaussian p(z) per object. Besides the challenge metrics, the table gives each ensemble's training time and splits the p(z) width into the members' disagreement ("model σ") and the spread over noise draws ("photometric σ").

Run this from the repo root on NERSC with the `qp` conda env as the kernel. Trained models, their training times and the predictions are saved in `TEST_DIR`, so a rerun skips what is done.

## Configuration

In [ ]:
import os

from rail.utils.path_utils import find_rail_file

import znn  # run from the repo root so this picks up the local source

# the configuration that degraded most in the noisytrain run
TASKSET, SIM, SCENARIO = 3, "cardinal", "10yr"
DATA_DIR = "../pz_data_challenge/scripts/public"  # as in the task-set notebooks

# Saved ensembles to re-evaluate: name -> candidate OUT_DIRs (the first that has the model file is used)
SAVED = {
    "baseline": ["outputs/znn_pretrained_popcosmos"],
}
# Ensembles trained here: name -> znn.train_from_catalog settings
TRAIN = {
    "kfold5": dict(N_MEMBERS=5, from_base=False),
    "frombase_kfold10": dict(N_MEMBERS=10, from_base=True),
}
TEST_DIR = f"outputs/frombase_test_taskset_{TASKSET}_{SIM}_{SCENARIO}"  # models trained here, their times, predictions

# as in the task-set notebooks
PRETRAINING = "popcosmos"
EPOCHS = 100  # max epochs per fully trained member, and of the base model
FINETUNE_EPOCHS = 10  # max epochs of each fine-tuned member
FINETUNE_LR = 1e-4  # fine-tuning learning rate (full training is at the Adam default, 1e-3)
VAL_FRAC = 0.2  # the held-out split (and the base model's early-stopping fraction)
SEED = 42
NOISE_SEED = 42  # seed of the training noise realizations
# as in pz_challenge_results.ipynb
N_SAMPLES = 50
PREDICT_SEED = 42

FILTER_DIR = find_rail_file("examples_data/estimation_data/data/FILTER")
BANDS = {
    "u": ("mag_u_lsst", f"{FILTER_DIR}/DC2LSST_u.res"),
    "g": ("mag_g_lsst", f"{FILTER_DIR}/DC2LSST_g.res"),
    "r": ("mag_r_lsst", f"{FILTER_DIR}/DC2LSST_r.res"),
    "i": ("mag_i_lsst", f"{FILTER_DIR}/DC2LSST_i.res"),
    "z": ("mag_z_lsst", f"{FILTER_DIR}/DC2LSST_z.res"),
    "y": ("mag_y_lsst", f"{FILTER_DIR}/DC2LSST_y.res"),
    "Y": ("mag_Y_roman", f"{FILTER_DIR}/roman_Y106.res"),
    "J": ("mag_J_roman", f"{FILTER_DIR}/roman_J129.res"),
    "H": ("mag_H_roman", f"{FILTER_DIR}/roman_H158.res"),
}
REF_BAND = "i"
FEATURE_CONFIG = znn.DEFAULT_FEATURE_CONFIG
ERRORS = {col: f"{col}_err" for col, _ in BANDS.values()}

In [ ]:
import json
import time

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import qp
from qp.metrics.pit import PIT
from sklearn.model_selection import train_test_split

from pz_data_challenge import scoring

os.makedirs(TEST_DIR, exist_ok=True)
MODEL_NAME = f"pz_challenge_taskset_{TASKSET}_{SIM}_pz_model_{SCENARIO}.pkl"

SPEC_SURVEYS = ["DEEP2_LSST", "DESI_BGS", "DESI_ELG_LOP", "DESI_LRG", "VVDSf02", "zCOSMOS"]


def reference_redshifts(train):
    # Reference redshift per training object, and the held-out subsets to score (as in pz_challenge_results.ipynb).
    z_spec = train["redshift"].to_numpy()
    if "COSMOS" not in train:  # task sets 1 and 2
        subsets = {"all": np.ones(len(train), dtype=bool)}
        if (train["mag_i_lsst"] < 23.5).all():
            return z_spec, subsets  # task set 1
        return z_spec, subsets | {"i > 23": train["mag_i_lsst"].to_numpy() > 23}  # task set 2

    is_spec = train[[c for c in SPEC_SURVEYS if c in train]].astype(bool).any(axis=1).to_numpy()
    is_cosmos_only = train["COSMOS"].astype(bool).to_numpy() & ~is_spec
    z_ref = np.where(is_cosmos_only, train["redshift_manyband"].to_numpy(), z_spec)
    assert np.isfinite(z_ref).all()
    return z_ref, {"all": np.ones(len(train), dtype=bool), "spec": is_spec, "COSMOS-only": is_cosmos_only}


train = znn.read_catalog(os.path.join(DATA_DIR, f"pz_challenge_taskset_{TASKSET}_{SIM}_training_{SCENARIO}.hdf5"))
z_ref, subsets = reference_redshifts(train)
idx_fit, idx_val = train_test_split(np.arange(len(train)), test_size=VAL_FRAC, random_state=SEED)
val = train.iloc[idx_val]
print(f"task set {TASKSET} {SIM} {SCENARIO}: {len(idx_fit)} objects to train on, {len(idx_val)} held out")

model_files = {}
for name, candidates in SAVED.items():
    found = [d for d in candidates if os.path.exists(os.path.join(d, MODEL_NAME))]
    assert found, f"no saved {name} model {MODEL_NAME} in {candidates}"
    model_files[name] = os.path.join(found[0], MODEL_NAME)
    print(f"{name}: {model_files[name]}")

## Train

Each ensemble in `TRAIN` is trained on the fit split with `znn.train_from_catalog` and saved in `TEST_DIR`, with its wall-clock training time. An ensemble already saved there is loaded instead.

In [ ]:
train_minutes = {}
for name, settings in TRAIN.items():
    model_files[name] = os.path.join(TEST_DIR, f"{name}_{MODEL_NAME}")
    time_file = os.path.join(TEST_DIR, f"{name}_train_time.json")
    if os.path.exists(model_files[name]) and os.path.exists(time_file):
        train_minutes[name] = json.load(open(time_file))["minutes"]
        print(f"{name}: already trained ({train_minutes[name]:.1f} min), {model_files[name]}")
        continue
    print(f"\n==================== {name}: {settings} ====================")
    start = time.time()
    trained_models, histories, base_history = znn.train_from_catalog(
        znn.build_model, train.iloc[idx_fit], z_ref[idx_fit], BANDS, REF_BAND, FEATURE_CONFIG, errors=ERRORS,
        rng=np.random.default_rng(NOISE_SEED), EPOCHS=EPOCHS, FINETUNE_EPOCHS=FINETUNE_EPOCHS, FINETUNE_LR=FINETUNE_LR,
        random_state=SEED, pretraining=PRETRAINING, **settings,
    )
    train_minutes[name] = (time.time() - start) / 60
    json.dump({"minutes": train_minutes[name]}, open(time_file, "w"))
    znn.save_ensemble_file(model_files[name], trained_models, BANDS, REF_BAND, FEATURE_CONFIG)
    print(f"{name}: trained in {train_minutes[name]:.1f} min")

    if base_history is not None:
        znn.plot_ensemble_losses([base_history], ylim=(0, 0.1))
        plt.suptitle(f"{name}: base model", y=1.08)
        plt.show()
    znn.plot_ensemble_losses(histories, ylim=(0, 0.1))
    plt.suptitle(name, y=1.08)
    plt.show()

## Predict the held-out objects

Every ensemble with the same `N_SAMPLES` noise draws (same seed). Cached in `TEST_DIR`.

In [ ]:
def heldout_predictions(name):
    # Per held-out object: p(z) mean and std, and the std split into members' disagreement and noise draws.
    cache = os.path.join(TEST_DIR, f"heldout_{name}.npz")
    if os.path.exists(cache):
        return dict(np.load(cache))
    trained_models, features = znn.load_ensemble_file(model_files[name])
    errors = {col: f"{col}_err" for col, _ in features["bands"].values()}
    pz, components = znn.predict_from_catalog(
        trained_models, val, **features, errors=errors, n_samples=N_SAMPLES, rng=np.random.default_rng(PREDICT_SEED),
        return_components=True,
    )
    out = {
        "mean": np.squeeze(pz.mean()),
        "std": np.squeeze(pz.std()),
        "model_std": components["model"],
        "photometric_std": components["photometric"],
    }
    np.savez(cache, **out)
    return out


ARMS = list(SAVED) + list(TRAIN)
preds = {}
for name in ARMS:
    print("held-out predictions:", name)
    preds[name] = heldout_predictions(name)

## Scores

The challenge's seven metrics (as in `pz_challenge_results.ipynb`, with the mean as the point estimate) and their tier points, per ensemble and held-out subset, plus:
- `width/scatter`: median p(z) σ/(1+z) over the biweight scatter; 1 for p(z)s as wide as the actual errors, below 1 for overconfident ones;
- `model σ`, `photometric σ`: median of the two parts of the p(z) σ (they add in quadrature). `model σ` measures how different the members are;
- `pit_outlier_direct`: the PIT outlier rate computed directly (the scored `outlier` comes from a spline fit that breaks down when PIT piles up at 0 or 1).

Lower is better for all metrics except the points. Remember the run-to-run noise of about 5% in scatter and 15% in KS.

In [ ]:
def scores(z, d, m):
    dz = (d["mean"][m] - z[m]) / (1 + z[m])
    bias, _, scatter, _, abs_outlier_rate = znn.get_biweight_mean_sigma_outlier(dz)
    pz = qp.Ensemble(qp.stats.norm, data={"loc": d["mean"][m, None], "scale": d["std"][m, None]})
    pit = PIT(pz, z[m])
    values = {
        "mean": bias,
        "std": scatter,
        "abs_outlier_rate": abs_outlier_rate,
        "ks": pit.evaluate_PIT_KS().statistic,
        "CvM": pit.evaluate_PIT_CvM().statistic,
        "ksamp": pit.evaluate_PIT_anderson_ksamp().statistic,
        "outlier": pit.evaluate_PIT_outlier_rate(),
    }
    points = sum(lo <= float(values[k]) <= hi for k, ranges in scoring.metric_dict.items() for lo, hi in ranges)
    extra = {
        "pit_outlier_direct": np.mean((pit.pit_samps < 0.0001) | (pit.pit_samps > 0.9999)),
        "width/scatter": np.median(d["std"][m] / (1 + z[m])) / scatter,
        "model σ": np.median(d["model_std"][m]),
        "photometric σ": np.median(d["photometric_std"][m]),
        "points": points,
    }
    return values | extra, pit.pit_samps


z_val = z_ref[idx_val]
rows, pits = {}, {}
for name in ARMS:
    for subset, mask in subsets.items():
        rows[(subset, name)], pits[(subset, name)] = scores(z_val, preds[name], mask[idx_val])

table = pd.DataFrame(rows).T
table.index.names = ["subset", "ensemble"]
n_max = sum(len(r) for r in scoring.metric_dict.values())
print(f"points out of {n_max}")
table.round(4)

In [ ]:
# the key numbers on all held-out objects, relative to the fully trained K-fold members, with training times
key = ["std", "abs_outlier_rate", "ks", "width/scatter", "model σ", "points"]
rel = table.xs("all", level="subset")[key].copy()
rel["train time [min]"] = pd.Series(train_minutes)
for col in ["std", "abs_outlier_rate", "ks", "model σ"]:
    rel[col + " vs kfold5"] = rel[col] / rel.loc["kfold5", col] - 1
rel.round(4)

## PIT

In [ ]:
pit_bins = np.linspace(0, 1, 26)
fig, axes = plt.subplots(1, len(subsets), figsize=(5 * len(subsets), 4), sharey=True, squeeze=False)
for ax, subset in zip(axes[0], subsets):
    for name in ARMS:
        ax.hist(pits[(subset, name)], pit_bins, density=True, histtype="step", lw=1.5,
                label=f"{name} (KS {table.loc[(subset, name), 'ks']:.3f})")
    ax.axhline(1, color="k", lw=0.8, ls=":")
    ax.set_title(f"{subset} ({subsets[subset][idx_val].sum()} objects)")
    ax.set_xlabel("PIT")
axes[0, 0].set_ylabel("density")
axes[0, 0].legend(fontsize=9, frameon=False)
fig.suptitle(f"task set {TASKSET} {SIM} {SCENARIO}, held-out", y=1.02)
fig.tight_layout()
plt.show()